# מדד המחירים לצרכן — מול טבלת ה-SCD

גרסה של `Untitled5.ipynb` (הורדה ושמירה) ו-`analysis.ipynb` (ניתוח מגמות) שעובדת מול `[cpi].[dbo].[foodChainPricesScd]`.

בטבלה הישנה `foodChainPrices` נשמרה שורה לכל פריט בכל סניף בכל הורדה. בטבלת ה-SCD נשמרת שורה רק כשהפריט משתנה, עם `validfrom` / `validto`.
כדי לשחזר את מה שהיה בטעינה מסוימת לוקחים את הגרסה שהייתה בתוקף באותו זמן:
`validfrom <= load_ts AND (validto > load_ts OR validto IS NULL)`.

1. הורדה, חילוץ ושמירה — `crawler` → `xml_to_csv` → `csv_to_sql`
2. בדיקת טעינה (במקום `upload_check.sql`)
3. מגמת מחירים חודשית לפי `items_names.xlsx`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import pyodbc

ROOT = Path.cwd()  # the market-crawler folder (where this notebook is)
SERVER = r'PCH80041449\MSSQLSERVER01'
DATABASE = 'cpi'
TABLE = '[cpi].[dbo].[foodChainPricesScd]'

ANALYST_DIR = Path(r'C:\Users\johny\Desktop\consumer_price_index')
ITEMS_FILE = ANALYST_DIR / 'items_names.xlsx'
FIRST_MONTH = '2026-06'        # base month; the first ratio is the month after it
PRICE_CAPS = {'עגבניה': 100}   # top_item -> prices >= cap are dropped as outliers

RUN_PIPELINE = False  # True = download, convert and load today's prices (takes a while)

venv_python = ROOT / '.venv' / 'Scripts' / 'python.exe'
PIPELINE_PYTHON = str(venv_python) if venv_python.exists() else sys.executable


def connect():
    return pyodbc.connect(driver='ODBC Driver 17 for SQL Server', server=SERVER,
                          database=DATABASE, trusted_connection='yes')

## 1. הורדה, חילוץ ושמירה

מחליף את פונקציות ההורדה ואת `try_or_none` של המחברת הישנה (אפשרויות נוספות ב-README):
- `crawler` — מוריד את קובץ ה-PriceFull האחרון של כל סניף. קוד יציאה 1 = רשת או קובץ נכשלו; השאר ממשיך לטעינה.
- `xml_to_csv` — ממזג את כל קבצי ה-XML של ההרצה ל-CSV אחד.
- `csv_to_sql` — טוען ל-SCD, רק שינויים נכתבים. סניף שההורדה שלו נכשלה נשאר עם המחירים הקודמים.

In [ ]:
def run(*args):
    """Run `python -m <args>` from the project folder and stream its output."""
    env = dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1')
    proc = subprocess.Popen([PIPELINE_PYTHON, '-m'] + list(args), cwd=ROOT, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding='utf-8', errors='replace')
    for line in proc.stdout:
        print(line, end='')
    print('exit code %d' % proc.wait())
    return proc.returncode


if RUN_PIPELINE:
    run('crawler')
    if run('xml_to_csv') == 0:
        run('csv_to_sql')

## 2. בדיקת טעינה

ב-SCD אין שורה לכל קובץ בכל יום, לכן ספירת הקבצים והסניפים לכל הורדה נלקחת מ-`data/<run>/file_list.csv`
(אותם `cnt_files` / `count_branchs` / `min_dt_file` / `max_dt_file` של `upload_check.sql`, ובנוסף כמה קבצים ירדו בפועל).

In [ ]:
columns = ['chain', 'chain_id', 'sub_chain', 'store_id', 'timestamp', 'file_name', 'local_path']
files = pd.concat([pd.read_csv(path, dtype=str).reindex(columns=columns).assign(run=path.parent.name)
                   for path in sorted((ROOT / 'data').glob('*/file_list.csv'))], ignore_index=True)
files['downloaded'] = files.local_path.fillna('') != ''
files['branch'] = files.chain_id + '-' + files.sub_chain + '-' + files.store_id
files['dt_file'] = files.timestamp.str[:10].str.replace('-', '')

upload_check = files.groupby(['chain', 'run']).agg(
    cnt_files=('file_name', 'nunique'),
    downloaded=('downloaded', 'sum'),
    count_branchs=('branch', 'nunique'),
    min_dt_file=('dt_file', 'min'),
    max_dt_file=('dt_file', 'max'),
).reset_index()
upload_check.pivot(index='chain', columns='run', values='count_branchs')

In [ ]:
upload_check.pivot(index='chain', columns='run', values='downloaded')

In [ ]:
upload_check.pivot(index='chain', columns='run', values='min_dt_file')

In [ ]:
upload_check.pivot(index='chain', columns='run', values='max_dt_file')

מה-SCD עצמו: כמה גרסאות נפתחו / השתנו / הוסרו בכל טעינה בשלושת החודשים האחרונים (`new` = פריט חדש בסניף,
`changed` = פריט שהשתנה, `removed` = פריט שירד מסניף שנטען), ומצב נוכחי לכל רשת.

In [ ]:
ACTIVITY_SQL = '''
SELECT chain, validfrom AS load_ts, 'opened' AS kind, COUNT_BIG(*) AS n
FROM {t} WHERE validfrom >= DATEADD(month, -3, GETDATE())
GROUP BY chain, validfrom
UNION ALL
SELECT chain, validto, CASE WHEN isdeleted = 1 THEN 'removed' ELSE 'changed' END, COUNT_BIG(*)
FROM {t} WHERE validto >= DATEADD(month, -3, GETDATE())
GROUP BY chain, validto, isdeleted
'''.format(t=TABLE)

CURRENT_SQL = '''
SELECT chain, chainid, COUNT(DISTINCT subchainid + '-' + storeid) AS branches,
       COUNT_BIG(*) AS items, MAX(validfrom) AS last_change
FROM {t} WHERE iscurrent = 1
GROUP BY chain, chainid
ORDER BY chain
'''.format(t=TABLE)

con = connect()
activity = pd.read_sql(ACTIVITY_SQL, con)
current = pd.read_sql(CURRENT_SQL, con)
con.close()

activity = (activity.pivot_table(index=['chain', 'load_ts'], columns='kind', values='n', aggfunc='sum', fill_value=0)
            .reindex(columns=['opened', 'changed', 'removed'], fill_value=0))
activity['new'] = activity.opened - activity.changed
activity[['new', 'changed', 'removed']]

In [ ]:
current

## 3. מגמת מחירים חודשית

אותה לוגיקה של `analysis.ipynb`:
1. פריטים רלוונטיים מ-`items_names.xlsx` (עד 10 שמות לכל `top_item` / `like`). `like = 0` — התאמה מדויקת, `like = 1` — `LIKE '%שם%'`.
   שם שמתאים לכמה פריטים מקבל את הראשון לפי (`top_item`, `like`), כמו ה-`CASE` המקורי.
2. מחיר > 0, ומחירים חריגים מסוננים לפי `PRICE_CAPS` (במקור: עגבניה < 100).
3. לכל טעינה מ-`FIRST_MONTH` והלאה — המחיר שהיה בתוקף; ממוצע חודשי לכל פריט × סניף × ברקוד.
4. יחס בין כל חודש לקודמו לכל פריט × סניף × ברקוד, ממוצע לכל `item`, פחות 1.

הבדלים מהמקור: החודש נקבע לפי זמן הטעינה ולא לפי התאריך בשם הקובץ (`dt_file`), וסניף שנכשל בהורדה ממשיך עם המחיר האחרון הידוע במקום להיעלם מאותה טעינה.

In [ ]:
items_names = pd.read_excel(ITEMS_FILE)
items_names = items_names[items_names.relevant == 1]
items_names = items_names.groupby(['top_item', 'relevant', 'like']).head(10).copy()
# order of the original CASE: (top_item, like) groups in sorted order, first match wins
items_names['priority'] = items_names.groupby(['top_item', 'like']).ngroup()
items_names['max_price'] = items_names.top_item.map(PRICE_CAPS)
items_names.groupby('top_item').itemname.count()

In [ ]:
TREND_SQL = '''
WITH loads AS (
    SELECT validfrom AS load_ts FROM {t} WHERE validfrom >= ?
    UNION
    SELECT validto FROM {t} WHERE validto >= ?
),
matches AS (
    SELECT v.chainid, v.subchainid, v.storeid, v.itemcode, v.itemprice, v.validfrom, v.validto,
           i.top_item, i.max_price, i.priority
    FROM {t} v JOIN #items i ON i.is_like = 0 AND v.itemname = i.itemname
    WHERE v.itemprice > 0 AND (v.validto IS NULL OR v.validto > ?)
    UNION ALL
    SELECT v.chainid, v.subchainid, v.storeid, v.itemcode, v.itemprice, v.validfrom, v.validto,
           i.top_item, i.max_price, i.priority
    FROM {t} v JOIN #items i ON i.is_like = 1 AND v.itemname LIKE N'%' + i.itemname + N'%'
    WHERE v.itemprice > 0 AND (v.validto IS NULL OR v.validto > ?)
),
versions AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY chainid, subchainid, storeid, itemcode, validfrom
                                 ORDER BY priority) AS rn
    FROM matches
)
SELECT v.top_item AS item, v.chainid, v.subchainid, v.storeid, v.itemcode,
       CONVERT(char(6), l.load_ts, 112) AS mnth, AVG(v.itemprice) AS itemprice
FROM versions v
JOIN loads l ON l.load_ts >= v.validfrom AND (v.validto IS NULL OR l.load_ts < v.validto)
WHERE v.rn = 1 AND (v.max_price IS NULL OR v.itemprice < v.max_price)
GROUP BY v.top_item, v.chainid, v.subchainid, v.storeid, v.itemcode, CONVERT(char(6), l.load_ts, 112)
'''.format(t=TABLE)

start = pd.Timestamp(FIRST_MONTH + '-01').to_pydatetime()
con = connect()
cursor = con.cursor()
cursor.execute('''
CREATE TABLE #items (
    itemname nvarchar(125) COLLATE DATABASE_DEFAULT NOT NULL,
    is_like bit NOT NULL,
    top_item nvarchar(125) COLLATE DATABASE_DEFAULT NOT NULL,
    priority int NOT NULL,
    max_price float NULL)''')
cursor.executemany('INSERT INTO #items VALUES (?, ?, ?, ?, ?)', [
    (str(r.itemname), int(r.like), str(r.top_item), int(r.priority),
     None if pd.isna(r.max_price) else float(r.max_price))
    for r in items_names.itertuples()])
prices = pd.read_sql(TREND_SQL, con, params=[start] * 4)
con.close()
prices.groupby(['item', 'mnth']).size().unstack()

In [ ]:
monthly = prices.pivot_table(index=['item', 'chainid', 'subchainid', 'storeid', 'itemcode'],
                             columns='mnth', values='itemprice')
months = sorted(monthly.columns)
if len(months) < 2:
    raise ValueError('need loads from at least two months since FIRST_MONTH, found: %s' % months)
ratios = pd.DataFrame({m: monthly[m] / monthly[p] for p, m in zip(months, months[1:])}, index=monthly.index)

pairs = ratios.groupby(level='item').count()  # branch x itemcode pairs behind each ratio
df = ratios.groupby(level='item').mean() - 1
df

In [ ]:
pairs

In [ ]:
df.to_excel(ANALYST_DIR / ('trend_%s.xlsx' % months[-1]))
df.to_clipboard()